# McNemar's Test: Is My Model Actually Better?

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 2 hours  
**Week 7 | Notebook 11 | Topic: Statistical Significance of Performance Differences**

---

## Real-World Analogy First

Imagine a hospital hires **two doctors** to diagnose 100 patients for a rare condition.

- **Doctor A** correctly diagnoses **85 patients**
- **Doctor B** correctly diagnoses **84 patients**

Should the hospital promote Doctor A? Is Doctor A *actually* better, or did they just get lucky on one patient?  

Now scale this to credit card fraud:
- **Model A** achieves **85.3% accuracy**
- **Model B** achieves **84.7% accuracy**

Is Model A genuinely superior? Or is this a 0.6% noise fluctuation that would vanish with a different test set?

**This notebook teaches you how to answer that question rigorously.**

---

## Learning Objectives

| # | Objective |
|---|---|
| 1 | Understand why accuracy differences are not automatically meaningful |
| 2 | Build McNemar's test from scratch and understand the contingency table |
| 3 | Verify implementation against `statsmodels` |
| 4 | Apply Bonferroni correction for multiple comparisons |
| 5 | Compute effect size — not just *whether* different, but *how much* |

---

## The Core Problem

When two classifiers are tested on the **same test set**, their predictions are **correlated** — both models see the same hard examples and the same easy ones. A standard t-test assumes independence, so it is wrong here. We need a test designed for **paired, dependent predictions**.

## Section 1: Setup and Fraud Dataset Creation

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# Sklearn classifiers and utilities
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# statsmodels for verification
from statsmodels.stats.contingency_tables import mcnemar as statsmodels_mcnemar

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')
np.random.seed(42)

print('All imports successful.')
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

In [ ]:
# ── Create Synthetic Credit Card Fraud Dataset ────────────────────────────────
# 2000 transactions, ~10% fraud rate
# Features are designed to have realistic fraud signals

np.random.seed(42)
n_samples = 2000
n_fraud = int(n_samples * 0.10)   # 200 fraudulent transactions
n_legit = n_samples - n_fraud     # 1800 legitimate

# --- Legitimate transactions ---
legit_amount          = np.random.exponential(scale=80, size=n_legit)      # typical spending
legit_time_since_last = np.random.exponential(scale=24, size=n_legit)      # hours
legit_num_txns_7d     = np.random.poisson(lam=5, size=n_legit)             # transactions/week
legit_credit_score    = np.random.normal(loc=700, scale=50, size=n_legit)  # FICO score
legit_account_age     = np.random.exponential(scale=1000, size=n_legit)    # days
legit_merchant_risk   = np.random.beta(a=2, b=8, size=n_legit)             # 0–1 risk score
legit_distance        = np.random.exponential(scale=20, size=n_legit)      # km from home
legit_is_intl         = (np.random.rand(n_legit) < 0.05).astype(int)       # 5% international
legit_is_night        = (np.random.rand(n_legit) < 0.15).astype(int)       # 15% night txns
legit_card_type       = np.random.choice([0, 1, 2], size=n_legit, p=[0.5, 0.35, 0.15])

# --- Fraudulent transactions (different distributions) ---
fraud_amount          = np.random.exponential(scale=300, size=n_fraud)     # larger amounts
fraud_time_since_last = np.random.exponential(scale=2, size=n_fraud)       # rapid succession
fraud_num_txns_7d     = np.random.poisson(lam=15, size=n_fraud)            # many transactions
fraud_credit_score    = np.random.normal(loc=600, scale=80, size=n_fraud)  # lower score
fraud_account_age     = np.random.exponential(scale=200, size=n_fraud)     # newer accounts
fraud_merchant_risk   = np.random.beta(a=6, b=4, size=n_fraud)             # riskier merchants
fraud_distance        = np.random.exponential(scale=200, size=n_fraud)     # far from home
fraud_is_intl         = (np.random.rand(n_fraud) < 0.40).astype(int)       # 40% international
fraud_is_night        = (np.random.rand(n_fraud) < 0.45).astype(int)       # 45% at night
fraud_card_type       = np.random.choice([0, 1, 2], size=n_fraud, p=[0.3, 0.45, 0.25])

# --- Assemble into DataFrame ---
feature_names = [
    'amount', 'time_since_last_txn', 'num_txns_7d', 'credit_score',
    'account_age_days', 'merchant_risk_score', 'distance_from_home',
    'is_international', 'is_night', 'card_type'
]

X_legit = np.column_stack([
    legit_amount, legit_time_since_last, legit_num_txns_7d, legit_credit_score,
    legit_account_age, legit_merchant_risk, legit_distance,
    legit_is_intl, legit_is_night, legit_card_type
])
X_fraud = np.column_stack([
    fraud_amount, fraud_time_since_last, fraud_num_txns_7d, fraud_credit_score,
    fraud_account_age, fraud_merchant_risk, fraud_distance,
    fraud_is_intl, fraud_is_night, fraud_card_type
])

X_all = np.vstack([X_legit, X_fraud])
y_all = np.array([0] * n_legit + [1] * n_fraud)

df = pd.DataFrame(X_all, columns=feature_names)
df['fraud'] = y_all

# Clip unrealistic values
df['credit_score'] = df['credit_score'].clip(300, 850)
df['amount']       = df['amount'].clip(0.01, 5000)

print(f'Dataset shape: {df.shape}')
print(f'Fraud rate: {df["fraud"].mean():.1%}')
print(f'\nClass distribution:\n{df["fraud"].value_counts()}')
df.head()

## Section 2: Train Four Models

We train four classifiers on 80% of the data and evaluate on the **same** 20% test set. This shared test set is what makes McNemar's test applicable.

In [ ]:
# ── Train-Test Split and Model Training ──────────────────────────────────────
# Stratified split preserves fraud rate in both train and test sets

X = df[feature_names].values
y = df['fraud'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y         # preserve class balance in both splits
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Test samples     : {X_test.shape[0]}')
print(f'Test fraud rate  : {y_test.mean():.1%}  (should be ~10%)')

# --- Define the four models as sklearn Pipelines with scaling ---
# Pipeline ensures the scaler is fit only on training data, preventing data leakage
models = {
    'LogReg'      : Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=500, random_state=42))]),
    'k-NN'        : Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5))]),
    'GaussianNB'  : Pipeline([('scaler', StandardScaler()), ('clf', GaussianNB())]),
    'DecisionTree': Pipeline([('scaler', StandardScaler()), ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))])
}

# --- Train all models and collect predictions on the SAME test set ---
predictions = {}   # model_name -> binary predictions array
accuracies  = {}   # model_name -> accuracy on test set

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc   = accuracy_score(y_test, preds)
    predictions[name] = preds
    accuracies[name]  = acc
    print(f'{name:15s}  accuracy = {acc:.4f}  ({acc*100:.2f}%)')

print('\nAccuracy differences (vs LogReg):')
base_acc = accuracies['LogReg']
for name, acc in accuracies.items():
    diff = acc - base_acc
    print(f'  {name:15s}: {diff:+.4f}  ← statistically meaningful?')

## Section 3: Understanding McNemar's Test

### The 2×2 Contingency Table

For two classifiers A and B tested on the same n samples, we build this table:

```
                    Model B Correct   Model B Wrong
Model A Correct  |     n₁₁         |     n₁₀       |
Model A Wrong    |     n₀₁         |     n₀₀       |
```

- **n₁₁**: Both models correct → tells us nothing about which is better
- **n₀₀**: Both models wrong → also tells us nothing about which is better  
- **n₁₀**: A correct, B wrong → evidence that A is better
- **n₀₁**: B correct, A wrong → evidence that B is better

**Key insight**: Only the *disagreement cells* (n₀₁ and n₁₀) carry information about which model is superior. If both models make the same mistakes, n₀₁ = n₁₀ and neither is better.

### The Test Statistic

$$\chi^2 = \frac{(|n_{01} - n_{10}| - 1)^2}{n_{01} + n_{10}}$$

The **−1** is a continuity correction (Yates' correction) for the discrete-to-continuous chi-squared approximation.

Under H₀ (both models make errors equally often), this follows a **χ² distribution with 1 degree of freedom**.

**H₀**: n₀₁ = n₁₀ (models are equivalent)  
**H₁**: n₀₁ ≠ n₁₀ (one model is significantly better)

In [ ]:
# ── McNemar's Test Implementation From Scratch ────────────────────────────────
# This function returns the full contingency table and test results.

def mcnemar_test(y_true, preds_a, preds_b, continuity_correction=True):
    """
    Perform McNemar's test comparing two classifiers on the same test set.

    Parameters
    ----------
    y_true              : array-like, true binary labels
    preds_a             : array-like, binary predictions from Model A
    preds_b             : array-like, binary predictions from Model B
    continuity_correction: bool, apply Yates' continuity correction (default True)

    Returns
    -------
    dict with keys: n11, n10, n01, n00, chi2, p_value, significant
    """
    y_true  = np.asarray(y_true)
    preds_a = np.asarray(preds_a)
    preds_b = np.asarray(preds_b)

    # Compute correctness vectors (True = correct prediction)
    correct_a = (preds_a == y_true)
    correct_b = (preds_b == y_true)

    # Build the 2×2 contingency table
    # n11: both correct | n10: A correct, B wrong
    # n01: B correct, A wrong | n00: both wrong
    n11 = np.sum( correct_a &  correct_b)   # both right
    n10 = np.sum( correct_a & ~correct_b)   # A right, B wrong
    n01 = np.sum(~correct_a &  correct_b)   # B right, A wrong
    n00 = np.sum(~correct_a & ~correct_b)   # both wrong

    assert n11 + n10 + n01 + n00 == len(y_true), 'Contingency table must sum to n'

    # Compute the test statistic
    # If n01 + n10 == 0, models agree perfectly → cannot reject H0
    denominator = n01 + n10
    if denominator == 0:
        return dict(n11=n11, n10=n10, n01=n01, n00=n00,
                    chi2=0.0, p_value=1.0, significant=False)

    if continuity_correction:
        # Yates' correction: subtract 1 from the absolute difference
        chi2_stat = (abs(n01 - n10) - 1.0) ** 2 / denominator
    else:
        chi2_stat = (n01 - n10) ** 2 / denominator

    # p-value from chi-squared distribution with df=1 (survival function = 1 - CDF)
    p_value = stats.chi2.sf(chi2_stat, df=1)

    return dict(
        n11=int(n11), n10=int(n10), n01=int(n01), n00=int(n00),
        chi2=round(chi2_stat, 4),
        p_value=round(p_value, 6),
        significant=(p_value < 0.05)
    )


# --- Demonstrate on LogReg vs k-NN ---
result = mcnemar_test(y_test, predictions['LogReg'], predictions['k-NN'])

print('McNemar\'s Test: LogReg vs k-NN')
print('=' * 45)
print(f'Contingency table:')
print(f'  n₁₁ (both correct)        = {result["n11"]:3d}')
print(f'  n₁₀ (LogReg✓, k-NN✗)     = {result["n10"]:3d}  ← LogReg advantage')
print(f'  n₀₁ (LogReg✗, k-NN✓)     = {result["n01"]:3d}  ← k-NN advantage')
print(f'  n₀₀ (both wrong)          = {result["n00"]:3d}')
print(f'\nTest statistic χ² = {result["chi2"]:.4f}')
print(f'p-value            = {result["p_value"]:.6f}')
print(f'Significant (α=0.05)? {result["significant"]}')

In [ ]:
# ── Verify Against statsmodels ────────────────────────────────────────────────
# statsmodels.stats.contingency_tables.mcnemar takes the 2×2 table as input

from statsmodels.stats.contingency_tables import mcnemar as sm_mcnemar

# Build the table in the format statsmodels expects
# table[i,j]: i=A_correct(0=wrong,1=right), j=B_correct(0=wrong,1=right)
# IMPORTANT: statsmodels uses [correct_a, correct_b] ordering:
#   table[1,1] = n11 (both correct)
#   table[1,0] = n10 (A correct, B wrong)
#   table[0,1] = n01 (A wrong, B correct)
#   table[0,0] = n00 (both wrong)

table_sm = np.array([
    [result['n00'], result['n01']],
    [result['n10'], result['n11']]
])

sm_result = sm_mcnemar(table_sm, exact=False, correction=True)

print('Verification: Our implementation vs statsmodels')
print('-' * 50)
print(f'Our χ²      : {result["chi2"]:.6f}')
print(f'statsmodels : {sm_result.statistic:.6f}')
print()
print(f'Our p-value      : {result["p_value"]:.6f}')
print(f'statsmodels p    : {sm_result.pvalue:.6f}')
print()

match_chi2 = abs(result['chi2'] - sm_result.statistic) < 1e-4
match_pval = abs(result['p_value'] - sm_result.pvalue) < 1e-4
print(f'Chi2 matches: {match_chi2}  |  p-value matches: {match_pval}')
print('\nVerification PASSED!' if (match_chi2 and match_pval) else 'WARNING: Mismatch!')

## Section 4: Why Not a t-test?

### The Independence Assumption Problem

A two-sample t-test compares means of **independent** groups. But when both models see the same test set, their errors are **correlated**:

- A complex, ambiguous transaction (e.g., a legitimate $3,000 jewelry purchase abroad at 2am) might fool **both** models
- A clear-cut fraud (obvious velocity attack) might be caught by **both** models

Using a t-test would ignore this shared difficulty structure and inflate the apparent variation, making the test either too conservative or too liberal.

**McNemar's test respects the paired structure**: it only looks at cases where the two models *disagree*, which is exactly the information that tells us which is better.

### Intuition Check

If n₁₀ = 20 (A beats B on 20 samples) and n₀₁ = 18 (B beats A on 18 samples), the difference is small and likely noise.

If n₁₀ = 40 and n₀₁ = 5, Model A is consistently better — that is unlikely to happen by chance.

## Section 5: All Pairwise Comparisons — C(4,2) = 6 Model Pairs

In [ ]:
# ── Pairwise McNemar Tests for All 6 Model Pairs ─────────────────────────────
# With 4 models there are C(4,2) = 6 unique pairs to compare

model_names = list(models.keys())
n_models    = len(model_names)

# Initialize matrices to store results
pval_matrix  = np.ones((n_models, n_models))   # p-values (diagonal = 1 by convention)
chi2_matrix  = np.zeros((n_models, n_models))  # chi-squared statistics
n10_matrix   = np.zeros((n_models, n_models), dtype=int)  # A-beats-B counts
n01_matrix   = np.zeros((n_models, n_models), dtype=int)  # B-beats-A counts

# Run all pairwise tests
print(f'Pairwise McNemar Tests (α = 0.05)\n{"="*60}')
print(f'{"Model A":15s}  {"Model B":15s}  {"n10":>5s}  {"n01":>5s}  {"χ²":>8s}  {"p-val":>10s}  Sig?')
print('-' * 75)

pair_results = []
for i in range(n_models):
    for j in range(i + 1, n_models):   # upper triangle only (avoid duplicates)
        name_a  = model_names[i]
        name_b  = model_names[j]
        res     = mcnemar_test(y_test, predictions[name_a], predictions[name_b])

        # Store in both directions for the heatmap (symmetric matrix)
        pval_matrix[i, j] = res['p_value']
        pval_matrix[j, i] = res['p_value']    # symmetric
        chi2_matrix[i, j] = res['chi2']
        chi2_matrix[j, i] = res['chi2']
        n10_matrix[i, j]  = res['n10']        # A-beats-B
        n01_matrix[i, j]  = res['n01']        # B-beats-A
        n10_matrix[j, i]  = res['n01']        # reverse direction: now B is "A"
        n01_matrix[j, i]  = res['n10']

        sig_marker = '***' if res['p_value'] < 0.001 else ('**' if res['p_value'] < 0.01 else ('*' if res['p_value'] < 0.05 else 'n.s.'))
        pair_results.append({
            'Model A': name_a, 'Model B': name_b,
            'n10': res['n10'], 'n01': res['n01'],
            'chi2': res['chi2'], 'p_value': res['p_value'],
            'Significant': res['significant']
        })
        print(f'{name_a:15s}  {name_b:15s}  {res["n10"]:5d}  {res["n01"]:5d}  {res["chi2"]:8.4f}  {res["p_value"]:10.6f}  {sig_marker}')

pairs_df = pd.DataFrame(pair_results)
print(f'\nSignificance: * p<0.05  ** p<0.01  *** p<0.001  n.s. = not significant')
print(f'\nNumber of significant pairs (α=0.05): {pairs_df["Significant"].sum()} / {len(pairs_df)}')

## Section 6: Bonferroni Correction for Multiple Comparisons

### The Multiple Comparisons Problem

We ran **6 tests** at α = 0.05. If all null hypotheses were true (all models equally good), we would *expect* 6 × 0.05 = **0.3 false positives** — about 1 in 3 experiments would produce a spurious "significant" result.

For 10 models (45 tests): expected false positives = 45 × 0.05 = **2.25** — almost guaranteed to find something fake!

### Bonferroni Correction

The simplest fix: divide α by the number of tests k.

$$\alpha_{\text{Bonferroni}} = \frac{\alpha}{k} = \frac{0.05}{6} \approx 0.0083$$

**Trade-off**: Bonferroni is conservative — it reduces false positives but increases false negatives (misses real differences). It controls the **Family-Wise Error Rate (FWER)**: the probability that *any* of the k tests is a false positive.

In [ ]:
# ── Apply Bonferroni Correction ───────────────────────────────────────────────
# k = 6 tests (C(4,2) = 6 pairwise comparisons)

n_comparisons     = len(pairs_df)           # k = 6
alpha_original    = 0.05
alpha_bonferroni  = alpha_original / n_comparisons

print(f'Multiple Comparisons Correction')
print(f'===============================\n')
print(f'Number of tests   : {n_comparisons}')
print(f'Original α        : {alpha_original:.4f}')
print(f'Bonferroni α      : {alpha_bonferroni:.6f}  (= 0.05 / {n_comparisons})')
print(f'Expected FP (raw) : {n_comparisons * alpha_original:.2f}  (≈ 0.3 false positives)')
print(f'Expected FP (Bon) : {n_comparisons * alpha_bonferroni:.2f}  (≈ 0.05 false positives)')
print()

# Annotate the pairs dataframe with Bonferroni significance
pairs_df['Sig_raw']       = pairs_df['p_value'] < alpha_original
pairs_df['Sig_Bonferroni'] = pairs_df['p_value'] < alpha_bonferroni

# Also compute Bonferroni-corrected p-values (multiply raw p by k, cap at 1)
pairs_df['p_bonferroni'] = (pairs_df['p_value'] * n_comparisons).clip(upper=1.0)

# Print comparison table
print(f'Pair-by-Pair Results (before vs after Bonferroni)\n')
print(f'{"Pair":35s}  {"p-value":>10s}  {"Raw Sig?":>10s}  {"p_Bonf":>10s}  {"Bonf Sig?":>10s}')
print('-' * 85)
for _, row in pairs_df.iterrows():
    pair_str = f'{row["Model A"]} vs {row["Model B"]}'
    print(f'{pair_str:35s}  {row["p_value"]:10.6f}  {str(row["Sig_raw"]):>10s}  {row["p_bonferroni"]:10.6f}  {str(row["Sig_Bonferroni"]):>10s}')

print(f'\nSignificant pairs (raw α=0.05)       : {pairs_df["Sig_raw"].sum()} / {n_comparisons}')
print(f'Significant pairs (Bonferroni α={alpha_bonferroni:.4f}): {pairs_df["Sig_Bonferroni"].sum()} / {n_comparisons}')

## Section 7: Visualizations

In [ ]:
# ── Heatmap of p-values for All Model Pairs ───────────────────────────────────
# Two heatmaps: raw p-values and Bonferroni-corrected p-values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('McNemar\'s Test: Pairwise p-values for Credit Card Fraud Classifiers',
             fontsize=14, fontweight='bold', y=1.02)

# --- Left: Raw p-values ---
ax1 = axes[0]
# Mask the diagonal (comparing a model to itself is meaningless)
mask = np.eye(n_models, dtype=bool)

# Create annotation matrix: show p-value + significance star
annot_raw = np.full((n_models, n_models), '', dtype=object)
for i in range(n_models):
    for j in range(n_models):
        if i != j:
            p = pval_matrix[i, j]
            star = '*' if p < alpha_original else ''
            annot_raw[i, j] = f'{p:.3f}{star}'

sns.heatmap(
    pval_matrix, ax=ax1, mask=mask,
    annot=annot_raw, fmt='', cmap='RdYlGn_r',
    vmin=0, vmax=0.1,
    xticklabels=model_names, yticklabels=model_names,
    cbar_kws={'label': 'p-value'},
    linewidths=1, linecolor='gray'
)
ax1.set_title(f'Raw p-values (α = {alpha_original})\n* = significant', fontsize=11)

# --- Right: Bonferroni-corrected p-values ---
ax2 = axes[1]
bonf_pval_matrix = np.clip(pval_matrix * n_comparisons, 0, 1)
np.fill_diagonal(bonf_pval_matrix, 1.0)  # diagonal = 1

annot_bonf = np.full((n_models, n_models), '', dtype=object)
for i in range(n_models):
    for j in range(n_models):
        if i != j:
            p = bonf_pval_matrix[i, j]
            star = '*' if p < alpha_original else ''
            annot_bonf[i, j] = f'{p:.3f}{star}'

sns.heatmap(
    bonf_pval_matrix, ax=ax2, mask=mask,
    annot=annot_bonf, fmt='', cmap='RdYlGn_r',
    vmin=0, vmax=1,
    xticklabels=model_names, yticklabels=model_names,
    cbar_kws={'label': 'Bonferroni-corrected p-value'},
    linewidths=1, linecolor='gray'
)
ax2.set_title(f'Bonferroni-corrected p-values (α = {alpha_bonferroni:.4f})\n* = significant after correction', fontsize=11)

plt.tight_layout()
plt.savefig('mcnemar_pvalue_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmaps saved.')

In [ ]:
# ── Visualise the Contingency Table for the Best Pair ────────────────────────
# Show the 2×2 table graphically for the most interesting comparison

# Find the pair with the smallest p-value (most significant difference)
best_pair_idx = pairs_df['p_value'].idxmin()
best_pair     = pairs_df.loc[best_pair_idx]
name_a        = best_pair['Model A']
name_b        = best_pair['Model B']

res_best = mcnemar_test(y_test, predictions[name_a], predictions[name_b])

# Build a nice contingency table for plotting
ct_data = np.array([
    [res_best['n11'], res_best['n10']],
    [res_best['n01'], res_best['n00']]
])

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(ct_data, cmap='Blues', aspect='auto')

# Annotations for each cell
cell_labels = [
    ['n₁₁ (both correct)', 'n₁₀ (A✓, B✗) ← A wins'],
    ['n₀₁ (A✗, B✓) ← B wins', 'n₀₀ (both wrong)']
]
colors_text = [['white', 'black'], ['black', 'white']]
for i in range(2):
    for j in range(2):
        count = ct_data[i, j]
        label = cell_labels[i][j]
        col   = 'white' if ct_data[i, j] > ct_data.max() * 0.6 else 'black'
        ax.text(j, i, f'{count}\n({label})', ha='center', va='center',
                fontsize=10, color=col, fontweight='bold')

ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels([f'{name_b}\nCorrect', f'{name_b}\nWrong'], fontsize=11)
ax.set_yticklabels([f'{name_a}\nCorrect', f'{name_a}\nWrong'], fontsize=11)
ax.set_title(
    f'McNemar\'s Contingency Table: {name_a} vs {name_b}\n'
    f'χ² = {res_best["chi2"]:.4f},  p = {res_best["p_value"]:.6f}',
    fontsize=12, fontweight='bold'
)

# Highlight the disagreement cells with border
for (i, j, color) in [(0, 1, 'red'), (1, 0, 'green')]:
    rect = plt.Rectangle((j - 0.5, i - 0.5), 1, 1,
                          fill=False, edgecolor=color, linewidth=3)
    ax.add_patch(rect)

plt.colorbar(im, ax=ax, label='Count')
plt.tight_layout()
plt.show()
print(f'\nOnly the RED and GREEN bordered cells determine the test outcome.')
print(f'n₁₀ (red) = {res_best["n10"]}  |  n₀₁ (green) = {res_best["n01"]}')

## Section 8: Effect Size — Not Just Whether, But How Much

Statistical significance tells us: **"Is the difference real?"**  
Effect size tells us: **"Is the difference practically important?"**

A highly significant test on a large dataset might reveal a 0.1% accuracy difference — real but irrelevant.

We compute:
1. **Raw accuracy difference** between models
2. **95% confidence interval** on that difference (using normal approximation for proportions)
3. **Practical significance**: does the CI exclude zero? Does it exclude a *meaningful* threshold (e.g., 1%)?

In [ ]:
# ── Effect Size: Accuracy Difference + Confidence Interval ────────────────────
# For each model pair, compute: delta accuracy + 95% CI
# Using normal approximation for proportion difference (valid for large n)

def accuracy_diff_ci(y_true, preds_a, preds_b, alpha=0.05):
    """
    Compute the difference in accuracy (A - B) and its confidence interval.
    Uses the standard error for the difference of two correlated proportions.

    Reference: Fleiss, Levin, Paik (2003) 'Statistical Methods for Rates and Proportions'
    """
    n = len(y_true)
    correct_a = (preds_a == y_true)
    correct_b = (preds_b == y_true)

    acc_a = correct_a.mean()       # accuracy of model A
    acc_b = correct_b.mean()       # accuracy of model B
    delta = acc_a - acc_b          # point estimate of difference

    # Standard error for correlated proportions (McNemar's SE)
    # SE² = [p_a*(1-p_a) + p_b*(1-p_b) - 2*(p11 - p_a*p_b)] / n
    p11 = (correct_a & correct_b).mean()    # proportion where both correct
    se  = np.sqrt((acc_a*(1-acc_a) + acc_b*(1-acc_b) - 2*(p11 - acc_a*acc_b)) / n)

    z_crit = stats.norm.ppf(1 - alpha / 2)   # 1.96 for 95% CI
    ci_lo  = delta - z_crit * se
    ci_hi  = delta + z_crit * se

    return dict(acc_a=acc_a, acc_b=acc_b, delta=delta, se=se, ci_lo=ci_lo, ci_hi=ci_hi)


# --- Compute for all 6 pairs ---
print('Effect Size Analysis: Accuracy Difference + 95% Confidence Interval')
print('=' * 75)
print(f'{"Pair":35s}  {"Δ Acc":>8s}  {"95% CI":>22s}  {"Excl. 0?":>9s}')
print('-' * 78)

effect_rows = []
for _, row in pairs_df.iterrows():
    name_a, name_b = row['Model A'], row['Model B']
    es = accuracy_diff_ci(y_test, predictions[name_a], predictions[name_b])
    excludes_zero = (es['ci_lo'] > 0) or (es['ci_hi'] < 0)

    pair_str = f'{name_a} vs {name_b}'
    ci_str   = f'[{es["ci_lo"]:+.4f}, {es["ci_hi"]:+.4f}]'
    print(f'{pair_str:35s}  {es["delta"]:+8.4f}  {ci_str:22s}  {str(excludes_zero):>9s}')

    effect_rows.append({
        'Model A': name_a, 'Model B': name_b,
        'Acc A': es['acc_a'], 'Acc B': es['acc_b'],
        'Delta': es['delta'],
        'CI_lo': es['ci_lo'], 'CI_hi': es['ci_hi'],
        'Excl_Zero': excludes_zero
    })

effect_df = pd.DataFrame(effect_rows)
print(f'\nNote: Δ = Acc(Model A) - Acc(Model B).  Positive → Model A is better.')

In [ ]:
# ── Forest Plot: Effect Sizes with Confidence Intervals ───────────────────────
# A forest plot makes it easy to see which differences are real and how large they are

fig, ax = plt.subplots(figsize=(10, 6))

pair_labels = [f'{r["Model A"]} vs {r["Model B"]}' for _, r in effect_df.iterrows()]
deltas      = effect_df['Delta'].values
ci_lo       = effect_df['CI_lo'].values
ci_hi       = effect_df['CI_hi'].values

# Error bar lengths (half-widths)
xerr_lo = deltas - ci_lo    # distance from point to lower CI
xerr_hi = ci_hi - deltas    # distance from point to upper CI

# Color code: red = CI excludes zero, gray = CI crosses zero
colors = ['#d62728' if (lo > 0 or hi < 0) else '#aec7e8'
          for lo, hi in zip(ci_lo, ci_hi)]

y_pos = range(len(pair_labels))
ax.errorbar(
    deltas, y_pos,
    xerr=[xerr_lo, xerr_hi],
    fmt='o', color='steelblue', ecolor='steelblue',
    elinewidth=2, capsize=5, markersize=8,
    label='Point estimate + 95% CI'
)

# Color the markers based on significance
for i, (d, c) in enumerate(zip(deltas, colors)):
    ax.scatter(d, i, color=c, s=80, zorder=5)

# Reference line at zero (no difference)
ax.axvline(0, color='black', linestyle='--', linewidth=1.5, label='No difference (H₀)', alpha=0.7)

# Annotations
for i, (d, lo, hi, excl) in enumerate(zip(deltas, ci_lo, ci_hi, effect_df['Excl_Zero'])):
    label = f'{d:+.3f}' + ('*' if excl else '')
    ax.annotate(label, (d, i), textcoords='offset points', xytext=(8, 4), fontsize=8)

ax.set_yticks(list(y_pos))
ax.set_yticklabels(pair_labels, fontsize=10)
ax.set_xlabel('Accuracy Difference (Model A − Model B)', fontsize=12)
ax.set_title('Forest Plot: Pairwise Accuracy Differences with 95% Confidence Intervals\n'
             'Red = CI excludes zero (statistically significant)  |  * marks significant pairs',
             fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.show()

## Section 9: The 5×2 Cross-Validated Paired t-test (Dietterich 1998)

### Motivation

McNemar's test uses a **single** train-test split, which means:
- The result depends on which 20% of samples ended up in the test set
- Small datasets amplify this sensitivity

**Dietterich's 5×2 cv paired t-test** addresses this by:
1. Repeating the following **5 times** (different random seeds):
   - Split data 50%/50% into folds 1 and 2
   - Train A on fold 1, test on fold 2 → difference d¹ᵢ
   - Train A on fold 2, test on fold 1 → difference d²ᵢ
2. Computing a t-statistic that accounts for the variance across splits

### Why It Is More Reliable

- Uses the full dataset (no single held-out set)
- Averages out the variance from specific splits
- Preferred for small-to-medium datasets where a single split is noisy

In [ ]:
# ── 5×2 Cross-Validated Paired t-test (Dietterich 1998) ──────────────────────
# We compare LogReg vs GaussianNB using this more robust test

def dietterich_5x2_cv_test(model_a, model_b, X, y, n_splits=5, random_state=0):
    """
    Dietterich's 5×2 cross-validated paired t-test.

    For each of 5 different random splits:
      - Split data 50/50 into two folds
      - Compute accuracy difference d_{i,1} (train on fold 1, test on fold 2)
      - Compute accuracy difference d_{i,2} (train on fold 2, test on fold 1)

    The test statistic t = d_{1,1} / sqrt(1/5 * sum_i(s_i^2))
    where s_i^2 = (d_{i,1} - p_i)^2 + (d_{i,2} - p_i)^2
    and p_i = (d_{i,1} + d_{i,2}) / 2
    """
    rng         = np.random.RandomState(random_state)
    n           = len(y)
    differences = []     # list of (d_i1, d_i2) for each split

    for split_idx in range(n_splits):
        # Random 50/50 split (un-stratified for simplicity)
        seed    = rng.randint(0, 10000)
        idx     = np.arange(n)
        np.random.seed(seed)
        np.random.shuffle(idx)
        mid     = n // 2
        fold1   = idx[:mid]
        fold2   = idx[mid:]

        # Fold A: train on fold1, evaluate on fold2
        model_a.fit(X[fold1], y[fold1])
        model_b.fit(X[fold1], y[fold1])
        acc_a_f1 = accuracy_score(y[fold2], model_a.predict(X[fold2]))
        acc_b_f1 = accuracy_score(y[fold2], model_b.predict(X[fold2]))
        d_i1     = acc_a_f1 - acc_b_f1    # positive → A better on this split

        # Fold B: train on fold2, evaluate on fold1
        model_a.fit(X[fold2], y[fold2])
        model_b.fit(X[fold2], y[fold2])
        acc_a_f2 = accuracy_score(y[fold1], model_a.predict(X[fold1]))
        acc_b_f2 = accuracy_score(y[fold1], model_b.predict(X[fold1]))
        d_i2     = acc_a_f2 - acc_b_f2

        differences.append((d_i1, d_i2))

    # Compute the test statistic
    # Variance estimate: mean of (d_i1 - p_i)^2 + (d_i2 - p_i)^2
    s2_list = []
    for d_i1, d_i2 in differences:
        p_i  = (d_i1 + d_i2) / 2.0
        s2_i = (d_i1 - p_i)**2 + (d_i2 - p_i)**2
        s2_list.append(s2_i)

    variance_estimate = np.mean(s2_list)

    # t-statistic uses the first split's d_{1,1} as numerator
    t_stat  = differences[0][0] / np.sqrt(variance_estimate + 1e-12)
    p_value = 2 * stats.t.sf(abs(t_stat), df=5)   # two-tailed, df=5

    return dict(
        t_statistic=round(t_stat, 4),
        p_value=round(p_value, 6),
        significant=(p_value < 0.05),
        differences=differences
    )


# --- Compare LogReg vs GaussianNB with both methods ---
# Re-create fresh model instances so fit history doesn't carry over
model_a_fresh = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=500, random_state=42))])
model_b_fresh = Pipeline([('scaler', StandardScaler()), ('clf', GaussianNB())])

result_5x2 = dietterich_5x2_cv_test(model_a_fresh, model_b_fresh, X, y)

# McNemar on same pair for comparison
result_mcn  = mcnemar_test(y_test, predictions['LogReg'], predictions['GaussianNB'])

print('Comparing Two Statistical Tests: LogReg vs GaussianNB')
print('=' * 60)
print(f'\n--- McNemar\'s Test (single 80/20 split) ---')
print(f'  χ² = {result_mcn["chi2"]:.4f},  p = {result_mcn["p_value"]:.6f}  Sig: {result_mcn["significant"]}')

print(f'\n--- Dietterich 5×2 CV Paired t-test ---')
print(f'  t  = {result_5x2["t_statistic"]:.4f},  p = {result_5x2["p_value"]:.6f}  Sig: {result_5x2["significant"]}')

print(f'\n5-split differences (LogReg acc − GaussNB acc):')
for i, (d1, d2) in enumerate(result_5x2['differences']):
    print(f'  Split {i+1}: d_1={d1:+.4f}, d_2={d2:+.4f}, mean={0.5*(d1+d2):+.4f}')

print(f'\nKey insight: 5×2 uses all data and multiple splits → more reliable for small/medium datasets.')

## Section 10: Summary and Key Takeaways

| Concept | Key Point |
|---|---|
| **Why McNemar's?** | Classifier errors on the same test set are correlated; McNemar's is designed for paired predictions |
| **Contingency table** | Only the n₀₁ and n₁₀ cells (disagreements) carry information about which model is better |
| **Multiple comparisons** | k tests at α=0.05 → expect k×0.05 false positives; Bonferroni divides α by k |
| **Effect size** | Significance alone is not enough; a 0.1% difference may be real but useless |
| **5×2 cv t-test** | More robust than McNemar's when a single split is noisy (small datasets) |
| **Practical rule** | A model difference is worth acting on only if: (1) p < α_corrected AND (2) CI excludes zero AND (3) effect size is practically relevant |

### When to Use What

- **McNemar's**: large test set (n > 500), simple and fast
- **5×2 cv paired t-test**: small-to-medium dataset, more reliable variance estimate
- **Bonferroni**: always, when comparing more than 2 models

## Self-Check Questions

Test your understanding before moving on.

---

**Q1.** Model A gets 89.2% accuracy, Model B gets 88.8% on a test set of n = 50. Is the difference likely significant? Why?

<details>
<summary>Show Answer</summary>

**Very unlikely to be significant.** With only n = 50 test samples, a 0.4% difference corresponds to less than 1 sample changing its prediction (0.004 × 50 = 0.2 samples). McNemar's test looks at n₀₁ and n₁₀ — the disagreement cells. With n = 50, we might have n₀₁ = 1, n₁₀ = 0 or n₀₁ = 1, n₁₀ = 1. These tiny counts yield enormous p-values. Rule of thumb: McNemar's requires n₀₁ + n₁₀ ≥ 25 for the chi-squared approximation to be reliable. With n = 50 total and a small accuracy gap, we almost certainly lack the statistical power to detect real differences. We would need a much larger test set or use an exact binomial test.

</details>

---

**Q2.** Why does McNemar's test only look at n₀₁ and n₁₀ (disagreement cells)? What information is in n₀₀ and n₁₁?

<details>
<summary>Show Answer</summary>

**n₁₁ (both correct) and n₀₀ (both wrong) tell us nothing about which model is better** — they are samples where both models perform identically. Any difference in model quality must come from samples where one model succeeds and the other fails. The disagreement cells n₁₀ (A correct, B wrong) and n₀₁ (B correct, A wrong) are direct evidence of relative advantage. Under H₀, we expect n₁₀ = n₀₁ (errors are symmetric). A large imbalance (e.g., n₁₀ = 30, n₀₁ = 5) means Model A is consistently winning on samples where they disagree. The agreement cells n₁₁ and n₀₀ simply reflect how many samples are "too easy" or "too hard" for both models — irrelevant to the comparison.

</details>

---

**Q3.** You compare 10 models pairwise using McNemar's test at α = 0.05. How many false positives do you expect? What correction should you apply?

<details>
<summary>Show Answer</summary>

With 10 models, there are C(10,2) = **45 pairwise tests**. At α = 0.05, the expected number of false positives (under H₀) is 45 × 0.05 = **2.25 false discoveries** — meaning even if all models are equally good, you will almost certainly find 2–3 "significant" pairs by chance alone.

**Bonferroni correction**: use α_corrected = 0.05 / 45 ≈ **0.0011** per test. This controls the Family-Wise Error Rate (FWER) — the probability that any test produces a false positive — at 5%. The trade-off is reduced power: real differences need larger n₁₀ − n₀₁ gaps to be detected. An alternative is the **Benjamini-Hochberg (FDR) correction**, which controls the False Discovery Rate and is less conservative when there are many real effects.

</details>